# A One-Day Introduction to Computational Neuroscience

## From a single Hodgkin–Huxley neuron to population coding

This notebook is adapted from the provided Brian2 simulations notebook. There are two projects here. The first part is introductory to the Brian functions that is common to both.

### Research pathways

Choose **one** project:

- **Project A — Hodgkin-Huxley neuron:** How does an ion-channel conductance affect excitability?
- **Project B — Population (LIF neurons):** How does connecitivy and ratio of excitatory to inhibitory neurons affect population activity

### Final deliverable

Prepare a short research summary containing:

- one research question;
- one testable hypothesis;
- a brief methods paragraph;
- two or three figures;
- three main findings;
- one limitation; and
- one follow-up experiment.

## Setup

This notebook requires Python, Brian2, NumPy, Matplotlib, and pandas.

When working in a new environment, install the packages with:

```python
%pip install brian2 numpy matplotlib pandas
```

Here at colab you only need to install brian2

Then restart the kernel if requested.

The notebook uses Brian2's NumPy code-generation target. It is slower than compiled code for large simulations, but it avoids requiring a C/C++ compiler and is convenient for a classroom exercise.

In [ ]:
%pip install brian2

In [ ]:
import brian2 as br

from brian2 import umetre, ufarad, cm, siemens, mV, ms, volt, amp, Hz, farad, nA, msiemens, second
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

br.defaultclock.dt = 0.025*ms

# Reproducibility
np.random.seed(7)
br.seed(7)

print("Brian2 is ready.")

# Part 1 — The basic pieces of a Brian2 experiment

A computational experiment usually contains five pieces:

1. **A model:** equations describing how state variables change.
2. **Parameters:** quantities such as a membrane time constant or conductance.
3. **An input:** injected current or synaptic activity.
4. **A simulation:** numerical integration over time.
5. **Measurements:** voltage traces, spike times, firing rates, or other summaries.

We will first use a simple leaky integrate-and-fire model. This is not a biophysical model of an action potential. It is a compact model for learning the Brian2 workflow.

## Brian2 building blocks used in this notebook

Brian2 represents a simulation using Python objects. A `NeuronGroup` contains the model and its neurons; monitors observe the group; and `run(...)` advances everything through simulated time.

| Brian2 name | What it does in plain language | Where it appears here |
|---|---|---|
| [`start_scope()`](https://brian2.readthedocs.io/en/stable/reference/brian2.core.magic.start_scope.html) | Starts a clean experiment. Objects created before this call are no longer automatically included in the next `run(...)`. This is especially useful when a notebook runs many independent trials. | At the start of each simulation function. |
| [`NeuronGroup`](https://brian2.readthedocs.io/en/stable/reference/brian2.groups.neurongroup.NeuronGroup.html) | Creates one neuron or a population of neurons that follows the supplied equations. The first argument is the number of neurons. | `NeuronGroup(1, ...)` creates one HH neuron; `NeuronGroup(n_neurons, ...)` creates the population. |
| Model-equation string | Defines differential equations, intermediate expressions, and parameters. The text after `:` gives the required physical unit; `: 1` means dimensionless. | `lif_eqs` and `HH_EQS`. |
| `threshold` | A Boolean condition used to declare that a spike occurred. | `v > 1` in the LIF model and `v > -40*mV` in the HH model. |
| `reset` | Specifies what happens immediately after a detected spike. | The LIF neuron is reset to `v = 0`. The HH neuron has **no reset** because its action-potential waveform emerges from its ion-channel equations. |
| `refractory` | Prevents the same action potential from being counted repeatedly while the voltage remains above the detection level. | In the HH model, the neuron remains refractory while `v > -40*mV`. |
| [`StateMonitor`](https://brian2.readthedocs.io/en/stable/reference/brian2.monitors.statemonitor.StateMonitor.html) | Records continuously changing variables at each time step, such as voltage, gating variables, or ionic currents. `record=True` records all neurons; a list of indices records only that subset. | `lif_voltage`, `state`, and `voltage`. |
| [`SpikeMonitor`](https://brian2.readthedocs.io/en/stable/reference/brian2.monitors.spikemonitor.SpikeMonitor.html) | Records discrete spike events. `.t` contains spike times, `.i` contains neuron indices, `.count` gives each neuron's spike count, and `.num_spikes` gives the total. | `lif_spikes` and `spikes`. |
| [`Synapses`](https://brian2.readthedocs.io/en/stable/reference/brian2.synapses.synapses.Synapses.html) | Represents connections from a presynaptic source to a postsynaptic target. `connect(p=...)` creates each eligible connection with the specified probability, and `on_pre` describes what happens when the source neuron spikes. | The recurrent connections in Project B. |
| [`SpikeGeneratorGroup`](https://brian2.readthedocs.io/en/stable/reference/brian2.input.spikegeneratorgroup.SpikeGeneratorGroup.html) | Emits spikes at times chosen in advance. It is useful for a precisely controlled input. | The brief trigger that starts Project B's network cascade. |
| [`PopulationRateMonitor`](https://brian2.readthedocs.io/en/stable/reference/brian2.monitors.ratemonitor.PopulationRateMonitor.html) | Records the instantaneous firing rate averaged across the population. The raw rate can be smoothed after the simulation for plotting. | The population-rate traces in Project B. |
| [`TimedArray`](https://brian2.readthedocs.io/en/stable/reference/brian2.input.timedarray.TimedArray.html) | Turns an array into a time-dependent input that can be called inside an equation, for example `stimulus(t)`. | The rectangular injected-current step. |
| [`run(duration)`](https://brian2.readthedocs.io/en/stable/reference/brian2.core.magic.run.html) | Numerically advances all currently visible Brian2 objects for the requested amount of biological time. | `run(100*ms)` and the simulation helper functions. |
| [`defaultclock.dt`](https://brian2.readthedocs.io/en/stable/reference/brian2.core.clocks.defaultclock.html) | Sets the numerical time step. A smaller value gives finer temporal resolution but requires more computation. | `0.025*ms` for the HH simulations. |

### How to read the first example

- `lif_eqs` defines how `v` changes and declares one adjustable variable, `drive`.
- `lif = NeuronGroup(2, ...)` makes two neurons that obey the same equation but can have different parameter values.
- `lif.drive = [0.8, 1.25]` assigns one value to each neuron.
- `StateMonitor(...)` records the continuous voltage trajectories.
- `SpikeMonitor(...)` records when each neuron crosses threshold.
- `run(100*ms)` performs the simulation; the recorded arrays are then available for plotting and counting.

Brian2 checks physical dimensions, so quantities such as `100*ms`, `0.08*nA`, and `-65*mV` include their units. See the official [Brian2 units guide](https://brian2.readthedocs.io/en/stable/user/units.html) for more detail.

In [ ]:
br.start_scope()

tau = 10*ms

# Leaky integrate and fire neuron
lif_eqs = '''
dv/dt = (drive - v)/tau : 1
drive : 1
'''

# two separate neurons
lif = br.NeuronGroup(
    2,
    lif_eqs,
    threshold='v > 1',
    reset='v = 0',
    method='exact'
)

# Neuron 0 receives subthreshold input; neuron 1 receives suprathreshold input.
lif.drive = [0.8, 1.25]

lif_voltage = br.StateMonitor(lif, 'v', record=True)
lif_spikes = br.SpikeMonitor(lif)

br.run(100*ms)

plt.figure(figsize=(9, 4))
for neuron_index in range(2):
    plt.plot(
        lif_voltage.t/ms,
        lif_voltage.v[neuron_index],
        label=f"Neuron {neuron_index}: drive={lif.drive[neuron_index]:.2f}"
    )
plt.plot(
    lif_spikes.t/ms,
    lif_spikes.i +.1,
    '|', c='k'
)
plt.axhline(1, linestyle='--', linewidth=1, label='Threshold')
plt.xlabel("Time (ms)")
plt.ylabel("Model voltage")
plt.title("A minimal Brian2 simulation")
plt.legend()
plt.show()

print("Spike counts:", np.asarray(lif_spikes.count))

### Check your understanding

Discuss with a partner:

1. Which line defines the model's differential equation?
2. Which object records the membrane voltage?
3. Which object records spikes?
4. Why does one neuron spike while the other does not?
5. What biological details are absent from this model?

The Hodgkin–Huxley model below adds voltage-dependent sodium and potassium channels, allowing the action potential to emerge from the equations rather than being imposed by a reset rule.

# Part 2 — Hodgkin–Huxley current-clamp experiment

The Hodgkin–Huxley (HH) neuron treats the membrane as a capacitor. Injected current can charge the membrane, while leak, sodium, and potassium currents flow through parallel ion channels. With the sign convention used in this notebook, the complete membrane equation is:

$$
C\frac{dV}{dt}
=
I_{\mathrm{stim}}
- g_{\mathrm{leak}}(V-E_{\mathrm{leak}})
- g_{\mathrm{Na}}m^3h(V-E_{\mathrm{Na}})
- g_{\mathrm{K}}n^4(V-E_{\mathrm{K}}).
$$

Here, \(C\) is membrane capacitance, \(V\) is membrane potential, each \(g\) is a conductance, and each \(E\) is an ion's reversal potential. The sodium and potassium terms can also be written as separate channel currents:

$$
I_{\mathrm{Na}} = g_{\mathrm{Na}}m^3h(V-E_{\mathrm{Na}}),
\qquad
I_{\mathrm{K}} = g_{\mathrm{K}}n^4(V-E_{\mathrm{K}}).
$$

The gates change according to the general equation

$$
\frac{dx}{dt}
=
\alpha_x(V)(1-x)-\beta_x(V)x,
\qquad x\in\{m,h,n\}.
$$

The gating variables range approximately from 0 to 1 and represent the fraction of gates in an available or open state:

- \(m\): rapid sodium activation;
- \(h\): sodium inactivation;
- \(n\): delayed potassium activation.

Therefore, \(m^3h\) and \(n^4\) act like voltage-dependent fractions of the maximal sodium and potassium conductances. In the code, the individual currents are first calculated as `I_leak`, `I_Na`, and `I_K`, then subtracted in `dv/dt`, exactly as in the expanded equation above.

We will perform a **current-clamp experiment**: inject a controlled current step and observe the membrane response. Unlike the LIF model, the HH voltage is not manually reset after a spike; depolarization and repolarization emerge from the changing sodium and potassium currents.

In [ ]:
# Cell geometry and passive properties
area = 20000*umetre**2
C_base = 1*ufarad*cm**-2 * area
g_leak = 5e-5*siemens*cm**-2 * area

# Reversal potentials
E_leak = -65*mV
E_K = -90*mV
E_Na = 50*mV

# Baseline maximal channel conductances
g_Na_base = 100*msiemens*cm**-2 * area
g_K_base = 30*msiemens*cm**-2 * area

# Voltage offset used in the original model
V_T = -63*mV

HH_EQS = '''
dv/dt = (I_stim - I_leak - I_Na - I_K)/C : volt

I_leak = g_leak*(v - E_leak) : amp
I_Na = g_Na*m**3*h*(v - E_Na) : amp
I_K = g_K*n**4*(v - E_K) : amp
I_stim = stimulus(t) : amp

dm/dt = alpha_m*(1-m) - beta_m*m : 1
dh/dt = alpha_h*(1-h) - beta_h*h : 1
dn/dt = alpha_n*(1-n) - beta_n*n : 1

alpha_m = 0.32*(mV**-1)*(13*mV-v+V_T) /
          (exp((13*mV-v+V_T)/(4*mV))-1)/ms : Hz
beta_m = 0.28*(mV**-1)*(v-V_T-40*mV) /
         (exp((v-V_T-40*mV)/(5*mV))-1)/ms : Hz

alpha_h = 0.128*exp((17*mV-v+V_T)/(18*mV))/ms : Hz
beta_h = 4/(1+exp((40*mV-v+V_T)/(5*mV)))/ms : Hz

alpha_n = 0.032*(mV**-1)*(15*mV-v+V_T) /
          (exp((15*mV-v+V_T)/(5*mV))-1)/ms : Hz
beta_n = 0.5*exp((10*mV-v+V_T)/(40*mV))/ms : Hz

C : farad (constant)
g_Na : siemens (constant)
g_K : siemens (constant)
'''

## Helper functions

The first function estimates resting values for the gating variables. This avoids beginning every simulation with all gates artificially set to zero.

The second function creates a current step.

The third function runs one HH neuron and returns voltage, gating-variable, current, and spike recordings.

In [ ]:
def hh_gate_steady_state(v0):
    """Return approximate steady-state m, h, and n at voltage v0."""
    v = float(v0/mV)
    vt = float(V_T/mV)

    alpha_m = 0.32*(13 - v + vt) / np.expm1((13 - v + vt)/4)
    beta_m = 0.28*(v - vt - 40) / np.expm1((v - vt - 40)/5)

    alpha_h = 0.128*np.exp((17 - v + vt)/18)
    beta_h = 4/(1 + np.exp((40 - v + vt)/5))

    alpha_n = 0.032*(15 - v + vt) / np.expm1((15 - v + vt)/5)
    beta_n = 0.5*np.exp((10 - v + vt)/40)

    m_inf = alpha_m/(alpha_m + beta_m)
    h_inf = alpha_h/(alpha_h + beta_h)
    n_inf = alpha_n/(alpha_n + beta_n)

    return m_inf, h_inf, n_inf


def make_step_current(
    amplitude,
    duration=200*ms,
    onset=50*ms,
    offset=150*ms,
    dt=0.025*ms
):
    """Create a TimedArray containing one rectangular current step."""
    n_samples = int(np.round(float(duration/dt)))
    values = np.zeros(n_samples)*nA

    onset_index = int(np.round(float(onset/dt)))
    offset_index = int(np.round(float(offset/dt)))
    values[onset_index:offset_index] = amplitude

    return br.TimedArray(values, dt=dt)


def simulate_single_hh(
    current_amplitude,
    gna_scale=1.0,
    gk_scale=1.0,
    duration=200*ms,
    onset=50*ms,
    offset=150*ms,
    dt=0.025*ms
):
    """Run a controlled current-clamp experiment on one HH neuron."""
    br.start_scope()
    br.defaultclock.dt = dt

    stimulus = make_step_current(
        current_amplitude,
        duration=duration,
        onset=onset,
        offset=offset,
        dt=dt
    )

    namespace = {
        'stimulus': stimulus,
        'g_leak': g_leak,
        'E_leak': E_leak,
        'E_K': E_K,
        'E_Na': E_Na,
        'V_T': V_T,
    }

    neuron = br.NeuronGroup(
        1,
        HH_EQS,
        threshold='v > -40*mV',
        refractory='v > -40*mV',
        method='exponential_euler',
        namespace=namespace
    )

    neuron.C = C_base
    neuron.g_Na = g_Na_base*gna_scale
    neuron.g_K = g_K_base*gk_scale
    neuron.v = E_leak

    m0, h0, n0 = hh_gate_steady_state(E_leak)
    neuron.m = m0
    neuron.h = h0
    neuron.n = n0

    state = br.StateMonitor(
        neuron,
        ['v', 'm', 'h', 'n', 'I_stim', 'I_Na', 'I_K'],
        record=True
    )
    spikes = br.SpikeMonitor(neuron)

    br.run(duration)
    return state, spikes

## Demonstration: three current-clamp conditions

We will compare:

- no injected current;
- a small subthreshold current;
- a suprathreshold current.

The exact threshold depends on the model parameters and numerical settings. The selected currents are intentionally small because the modeled membrane area is also small.

In [ ]:
demo_conditions = {
    "No current": 0*nA,
    "Subthreshold": 0.020*nA,
    "Suprathreshold": 0.080*nA,
}

demo_results = {}

for label, amplitude in demo_conditions.items():
    demo_results[label] = simulate_single_hh(amplitude)
    print(
        f"{label:15s} | current={float(amplitude/nA):.3f} nA "
        f"| spikes={demo_results[label][1].num_spikes}"
    )

In [ ]:
plt.figure(figsize=(10, 5))

for label, (state, spikes) in demo_results.items():
    plt.plot(state.t/ms, state.v[0]/mV, label=label)

plt.axvspan(50, 150, alpha=0.12, label="Current step")
plt.axhline(-40, linestyle='--', linewidth=1, label="Spike-detection level")
plt.xlabel("Time (ms)")
plt.ylabel("Membrane potential (mV)")
plt.title("HH neuron responses to controlled current steps")
plt.legend()
plt.show()

In [ ]:
state_super, spikes_super = demo_results["Suprathreshold"]

plt.figure(figsize=(10, 4))
plt.plot(state_super.t/ms, state_super.m[0], label="m: Na activation")
plt.plot(state_super.t/ms, state_super.h[0], label="h: Na inactivation")
plt.plot(state_super.t/ms, state_super.n[0], label="n: K activation")
plt.xlabel("Time (ms)")
plt.ylabel("Gate value")
plt.title("Voltage-dependent gating variables")
plt.legend()
plt.show()

plt.figure(figsize=(10, 4))
plt.plot(state_super.t/ms, state_super.I_Na[0]/nA, label="Sodium current")
plt.plot(state_super.t/ms, state_super.I_K[0]/nA, label="Potassium current")
plt.plot(state_super.t/ms, state_super.I_stim[0]/nA, label="Injected current")
plt.axhline(0, linewidth=1)
plt.xlabel("Time (ms)")
plt.ylabel("Current (nA)")
plt.title("Ionic and injected currents")
plt.legend()
plt.show()

### Interpret the HH demonstration

Write brief answers before beginning a project:

1. During which phase of the spike does sodium activation rise?
2. Why does sodium current not remain large throughout the spike?
3. Why does potassium activation peak later than sodium activation?
4. Which current is mainly responsible for repolarization?
5. How is the suprathreshold response different from the subthreshold response?
6. What measurement could be used to define neuronal excitability?

---

# Part 3 — Choose one research project

A useful research project changes **one main factor**, holds other conditions constant, and measures a clearly defined response.

Before running a large sweep, write:

- your independent variable;
- your dependent variable;
- your control condition;
- your hypothesis and biological reasoning.

# Project A — Single-neuron excitability

## Project picture

<div style="display:flex; align-items:center; justify-content:center; gap:16px; flex-wrap:wrap; margin:14px 0 22px 0;">
  <div style="padding:10px 16px; border:2px solid #4C78A8; border-radius:8px; background:#EEF5FB; text-align:center;">
    <strong>Injected current</strong><br><code>I_stim</code>
  </div>
  <div style="font-size:28px; color:#555;">→</div>
  <div style="width:145px; height:100px; border:3px solid #F58518; border-radius:50%; background:#FFF3E6; display:flex; align-items:center; justify-content:center; text-align:center;">
    <span><strong>One HH neuron</strong><br>change <code>g_Na</code><br>or <code>g_K</code></span>
  </div>
  <div style="font-size:28px; color:#555;">→</div>
  <div style="padding:10px 16px; border:2px solid #54A24B; border-radius:8px; background:#EFF8EE; text-align:center;">
    <strong>Measure response</strong><br>spikes · rate · latency
  </div>
</div>

Keep the current protocol fixed, repeat the same virtual experiment at several conductance values, and compare the response of the **same model neuron** across conditions.

## Research question

**How does changing a maximal ion-channel conductance alter the excitability of one Hodgkin–Huxley neuron?**

Choose one channel:

- maximal sodium conductance, $g_{\mathrm{Na}}$; or
- maximal potassium conductance, $g_{\mathrm{K}}$.

Compare several conductance scales, such as:

$$
0.75,\quad 1.00,\quad 1.25
$$

times the baseline value.

## Suggested measurements

- spike count;
- firing rate during the current step;
- first-spike latency;
- rheobase, defined here as the smallest tested current producing at least one spike;
- optional: peak voltage or spike width.

## Example hypotheses

**Sodium version:** Increasing $g_{\mathrm{Na}}$ will reduce rheobase because regenerative inward current becomes stronger.

**Potassium version:** Increasing $g_{\mathrm{K}}$ will reduce sustained firing or increase rheobase because outward current more strongly opposes depolarization.

Rewrite the hypothesis in your own words and include a mechanistic explanation.

In [ ]:
def summarize_single_trial(
    state,
    spikes,
    onset=50*ms,
    offset=150*ms
):
    """Calculate a compact set of single-neuron response measurements."""
    spike_times = np.asarray(spikes.t/ms)
    onset_ms = float(onset/ms)
    offset_ms = float(offset/ms)

    during_step = (
        (spike_times >= onset_ms) &
        (spike_times < offset_ms)
    )
    step_spikes = spike_times[during_step]

    step_duration_seconds = float((offset - onset)/second)
    firing_rate_hz = len(step_spikes)/step_duration_seconds

    if len(step_spikes):
        first_spike_latency_ms = step_spikes[0] - onset_ms
    else:
        first_spike_latency_ms = np.nan

    state_times = np.asarray(state.t/ms)
    state_mask = (
        (state_times >= onset_ms) &
        (state_times < offset_ms)
    )
    peak_voltage_mV = np.max(np.asarray(state.v[0]/mV)[state_mask])

    return {
        'spike_count': len(step_spikes),
        'firing_rate_Hz': firing_rate_hz,
        'first_spike_latency_ms': first_spike_latency_ms,
        'peak_voltage_mV': peak_voltage_mV,
    }


# Try one condition before running the full sweep.
trial_state, trial_spikes = simulate_single_hh(
    current_amplitude=0.06*nA,
    gna_scale=1.0,
    gk_scale=1.0
)

summarize_single_trial(trial_state, trial_spikes)

## Run the conductance and current sweep

Edit `CHANNEL_TO_CHANGE` to `"Na"` or `"K"`.

You may also change the conductance scales and current values after inspecting the first result. Use a range that includes:

- currents producing no spikes;
- currents near threshold; and
- currents producing repeated spikes.

In [ ]:
# run simulations

In [ ]:
plot results

## Project A analysis prompts

1. Did changing the conductance alter rheobase?
2. Did it alter firing rate at every current, or mainly near threshold?
3. Did first-spike latency change in the direction you predicted?
4. Is the effect monotonic across all conductance values?
5. Which voltage or current traces help explain the result?
6. Does your evidence support the original hypothesis?
7. What biological or modeling limitation should be acknowledged?

### Minimum Project A figures

1. Voltage traces for at least three conductance conditions at the same current.
2. Firing rate versus injected current.
3. Rheobase or first-spike latency versus conductance.

### Optional extension

Measure spike width, after-hyperpolarization depth, or sodium/potassium current peaks.

# Project B — Recurrent excitatory–inhibitory LIF network

## Project picture

<div style="display:flex; align-items:center; justify-content:center; gap:14px; flex-wrap:wrap; margin:14px 0 22px 0;">
  <div style="padding:10px 14px; border:2px solid #7A5195; border-radius:8px; background:#F5EFF8; text-align:center;">
    <strong>Brief trigger</strong><br>a few neurons spike once
  </div>
  <div style="font-size:28px; color:#555;">→</div>
  <div style="padding:10px 14px; border:2px solid #666; border-radius:12px; background:#FAFAFA; text-align:center;">
    <strong>Connected LIF network</strong><br>
    <span style="display:inline-block; width:31px; height:31px; line-height:31px; margin:5px 2px 0; border-radius:50%; background:#8ECAE6;">E</span>
    <span style="display:inline-block; width:31px; height:31px; line-height:31px; margin:5px 2px 0; border-radius:50%; background:#8ECAE6;">E</span>
    <span style="display:inline-block; width:31px; height:31px; line-height:31px; margin:5px 2px 0; border-radius:50%; background:#FFB4A2;">I</span>
    <span style="display:inline-block; width:31px; height:31px; line-height:31px; margin:5px 2px 0; border-radius:50%; background:#8ECAE6;">E</span><br>
    <small>vary connectivity and the E:I ratio</small>
  </div>
  <div style="font-size:28px; color:#555;">→</div>
  <div style="padding:10px 14px; border:2px solid #54A24B; border-radius:8px; background:#EFF8EE; text-align:center;">
    <strong>Network outcome</strong><br>dies · persists · grows
  </div>
</div>

Every neuron is a leaky integrate-and-fire (LIF) neuron. Excitatory neurons increase the membrane variable of their targets when they spike; inhibitory neurons decrease it. The neurons are connected randomly, so the **connection probability** \(p\) controls how many directed synapses are expected.

The membrane variable of each neuron decays toward rest between synaptic events:

$$
\tau_m\frac{dv_i}{dt}=-v_i.
$$

When \(v_i>1\), the neuron emits a spike, resets to \(v_i=0\), and briefly becomes refractory. A presynaptic spike causes the postsynaptic update

$$
v_j \leftarrow
\begin{cases}
v_j + J_{\mathrm{E}}, & \text{excitatory presynaptic neuron},\\
v_j - J_{\mathrm{I}}, & \text{inhibitory presynaptic neuron}.
\end{cases}
$$

## Research question

**How do connection probability and the ratio of excitatory to inhibitory neurons determine whether activity triggered in a recurrent LIF network dies out, persists, or grows rapidly?**

The project varies:

- **connection probability** \(p\): the probability that an eligible pair of neurons is connected; and
- **excitatory fraction** \(f_{\mathrm{E}}\): the fraction of neurons that are excitatory. The remainder are inhibitory.

Hold population size, synaptic strength, trigger size, simulation duration, and random seed constant while comparing conditions.

## Example hypotheses

1. Increasing connection probability will make each spike influence more neurons, increasing the likelihood that the cascade persists or grows.
2. Increasing the excitatory fraction will shift the network toward sustained or rapidly growing activity, whereas stronger inhibitory representation will make the cascade more likely to die out.
3. Intermediate connectivity and E:I balance may produce a long but non-saturating, critical-like cascade.

In [ ]:
def simulate_recurrent_lif(
    n_neurons=200,
    excitatory_fraction=0.80,
    connection_probability=0.12,
    duration=200*ms,
    trigger_time=10*ms,
    trigger_size=20,
    tau_m=20*ms,
    refractory_period=2*ms,
    excitatory_jump=0.22,
    inhibitory_strength=4.0,
    trigger_jump=1.05,
    synaptic_delay=1*ms,
    dt=0.1*ms,
    seed_value=7
):
    # Run a recurrent E/I network of LIF neurons after one brief trigger.
    br.start_scope()
    br.defaultclock.dt = dt

    # Set both random-number generators for a reproducible network.
    np.random.seed(seed_value)
    br.seed(seed_value)

    n_excitatory = int(round(n_neurons*excitatory_fraction))
    n_inhibitory = n_neurons - n_excitatory

    if not 0 < n_excitatory < n_neurons:
        raise ValueError("The network must contain both E and I neurons.")
    if trigger_size > n_excitatory:
        raise ValueError("trigger_size must not exceed the number of E neurons.")

    recurrent_lif_eqs = '''
    dv/dt = -v/tau_m : 1 (unless refractory)
    synaptic_sign : 1 (constant)
    '''

    population = br.NeuronGroup(
        n_neurons,
        recurrent_lif_eqs,
        threshold='v > 1',
        reset='v = 0',
        refractory=refractory_period,
        method='exact',
        namespace={'tau_m': tau_m}
    )

    # The first neurons are excitatory; the remaining neurons are inhibitory.
    # Inhibitory events are stronger so that an 80:20 network can be balanced.
    population.synaptic_sign[:n_excitatory] = 1
    population.synaptic_sign[n_excitatory:] = -inhibitory_strength
    population.v = '0.05*rand()'

    recurrent = br.Synapses(
        population,
        population,
        on_pre='v_post += excitatory_jump*synaptic_sign_pre',
        delay=synaptic_delay,
        namespace={'excitatory_jump': excitatory_jump}
    )
    recurrent.connect(
        condition='i != j',
        p=connection_probability
    )

    # A separate spike source makes the first trigger_size excitatory neurons
    # receive one simultaneous suprathreshold input.
    trigger_indices = np.arange(trigger_size)
    trigger_times = np.full(
        trigger_size,
        float(trigger_time/ms)
    )*ms
    trigger = br.SpikeGeneratorGroup(
        trigger_size,
        trigger_indices,
        trigger_times
    )
    trigger_connections = br.Synapses(
        trigger,
        population,
        on_pre='v_post += trigger_jump',
        namespace={'trigger_jump': trigger_jump}
    )
    trigger_connections.connect(j='i')

    spikes = br.SpikeMonitor(population)
    population_rate = br.PopulationRateMonitor(population)

    br.run(duration)

    metadata = {
        'n_neurons': n_neurons,
        'n_excitatory': n_excitatory,
        'n_inhibitory': n_inhibitory,
        'excitatory_fraction': excitatory_fraction,
        'connection_probability': connection_probability,
        'n_connections': len(recurrent),
        'duration_ms': float(duration/ms),
        'trigger_time_ms': float(trigger_time/ms),
        'trigger_size': trigger_size,
    }
    return spikes, population_rate, metadata


def summarize_recurrent_activity(
    spikes,
    metadata,
    bin_size=5*ms
):
    # Summarize the post-trigger cascade using deliberately simple measures.
    spike_times_ms = np.asarray(spikes.t/ms)
    spike_indices = np.asarray(spikes.i)

    trigger_time_ms = metadata['trigger_time_ms']
    duration_ms = metadata['duration_ms']
    bin_size_ms = float(bin_size/ms)

    post_trigger = spike_times_ms >= trigger_time_ms
    times = spike_times_ms[post_trigger]
    indices = spike_indices[post_trigger]

    bin_edges = np.arange(
        trigger_time_ms,
        duration_ms + bin_size_ms,
        bin_size_ms
    )
    bin_counts, bin_edges = np.histogram(times, bins=bin_edges)
    bin_centres_ms = bin_edges[:-1] + bin_size_ms/2

    active_bins = np.flatnonzero(bin_counts > 0)
    if len(active_bins):
        active_duration_ms = (
            bin_edges[active_bins[-1] + 1] -
            bin_edges[active_bins[0]]
        )
    else:
        active_duration_ms = 0.0

    current_counts = bin_counts[:-1]
    next_counts = bin_counts[1:]
    valid_transitions = current_counts > 0

    if np.any(valid_transitions):
        branching_ratio = (
            np.sum(next_counts[valid_transitions]) /
            np.sum(current_counts[valid_transitions])
        )
    else:
        branching_ratio = np.nan

    total_spikes = len(times)
    halfway_time_ms = trigger_time_ms + (
        duration_ms - trigger_time_ms
    )/2
    late_spike_fraction = (
        np.mean(times >= halfway_time_ms)
        if total_spikes else 0.0
    )

    spikes_per_neuron = total_spikes/metadata['n_neurons']



    return {
        'post_trigger_spikes': total_spikes,
        'spikes_per_neuron': spikes_per_neuron,
        'fraction_neurons_active': (
            len(np.unique(indices))/metadata['n_neurons']
            if total_spikes else 0.0
        ),
        'active_duration_ms': active_duration_ms,
        'branching_ratio': branching_ratio,
        'late_spike_fraction': late_spike_fraction,
        'bin_centres_ms': bin_centres_ms,
        'bin_counts': bin_counts,
    }

## Compare three starting conditions

The following parameter combinations are intended to begin in different regions of the network's behavior:

- sparse and inhibition-dominated;
- near the transition between dying and sustained activity; and
- dense and excitation-dominated.

Because connectivity is random and the transition can be sharp, the exact result may differ slightly across Brian2 versions or random seeds. If two conditions look identical, adjust one connection probability by \(0.01\)–\(0.03\).

In [ ]:
example_conditions = {
    "Sparse / more inhibition": {
        'excitatory_fraction': 0.80,
        'connection_probability': 0.18,
    },
    "Near-transition candidate": {
        'excitatory_fraction': 0.83,
        'connection_probability': 0.22,
    },
    "Dense / more excitation": {
        'excitatory_fraction': 0.9,
        'connection_probability': 0.3,
    },
}

network_examples = {}

for label, parameters in example_conditions.items():
    spikes, rate, metadata = simulate_recurrent_lif(
        **parameters,
        seed_value=7
    )
    summary = summarize_recurrent_activity(
        spikes,
        metadata
    )
    network_examples[label] = {
        'spikes': spikes,
        'rate': rate,
        'metadata': metadata,
        'summary': summary,
    }

    print(
        f"{label:27s} | "
        f"E:I={metadata['n_excitatory']}:{metadata['n_inhibitory']} | "
        f"p={metadata['connection_probability']:.2f} | "
        f"spikes={summary['post_trigger_spikes']:5d} | "
        f"duration={summary['active_duration_ms']:5.1f} ms | "
        f"sigma={summary['branching_ratio']:.2f}"
    )

In [ ]:
fig, axes = plt.subplots(
    len(network_examples),
    2,
    figsize=(13, 10),
    sharex=True
)

for row, (label, result) in enumerate(network_examples.items()):
    spikes = result['spikes']
    rate = result['rate']
    metadata = result['metadata']
    n_excitatory = metadata['n_excitatory']

    spike_times = np.asarray(spikes.t/ms)
    spike_indices = np.asarray(spikes.i)
    is_excitatory = spike_indices < n_excitatory

    axes[row, 0].scatter(
        spike_times[is_excitatory],
        spike_indices[is_excitatory],
        s=5,
        color='#219EBC',
        label='Excitatory'
    )
    axes[row, 0].scatter(
        spike_times[~is_excitatory],
        spike_indices[~is_excitatory],
        s=5,
        color='#E76F51',
        label='Inhibitory'
    )
    axes[row, 0].axvline(
        metadata['trigger_time_ms'],
        color='black',
        linestyle='--',
        linewidth=1
    )
    axes[row, 0].axhline(
        n_excitatory - 0.5,
        color='grey',
        linewidth=1
    )
    axes[row, 0].set_ylabel("Neuron index")
    axes[row, 0].set_title(f"{label}: raster")

    smoothed_rate = rate.smooth_rate(
        window='gaussian',
        width=3*ms
    )
    axes[row, 1].plot(
        rate.t/ms,
        smoothed_rate/Hz,
        color='#6A4C93'
    )
    axes[row, 1].axvline(
        metadata['trigger_time_ms'],
        color='black',
        linestyle='--',
        linewidth=1
    )
    axes[row, 1].set_ylabel("Population rate (Hz)")
    axes[row, 1].set_title(f"{label}: smoothed rate")

axes[-1, 0].set_xlabel("Time (ms)")
axes[-1, 1].set_xlabel("Time (ms)")
axes[0, 0].legend(loc='upper right')
plt.tight_layout()
plt.show()

## Sweep connection probability and E:I ratio

The sweep below varies both independent variables while keeping the trigger and synaptic weights fixed. The same seed is reused so results are reproducible. A single graph realization can still be unusual, so repeating the sweep with several seeds is a useful extension.

Start with five values of each variable. If runtime is limited, reduce either list to three values while developing the analysis.

In [ ]:
# Run simulation

In [ ]:
# plot results

## Project B analysis prompts

1. How does increasing connection probability change total spiking and cascade duration?
2. How does increasing the excitatory fraction change the same outcomes?
3. Which conditions are subcritical-like, critical-like candidates, or high/self-sustained?
4. Can two networks have similar branching-ratio estimates but very different raster plots or total spike counts?
5. Does the transition occur gradually, or does a small parameter change cause a large shift?
6. How does the stronger inhibitory synaptic jump affect interpretation of the E:I ratio?
7. Does your evidence support the original hypotheses?
8. Why is this exercise insufficient to establish true neural criticality?

### Minimum Project B figures

1. Raster plots and population-rate traces for three contrasting conditions.
2. A heatmap showing activity magnitude or duration across connection probability and E:I ratio.
3. Approximate branching ratio versus connection probability.

### Optional extensions

- Repeat each parameter combination with several random graph seeds and show uncertainty.
- Vary excitatory or inhibitory synaptic strength while holding the E:I ratio constant.
- Introduce heterogeneous synaptic delays to reduce synchronous waves.
- Add weak background input and test whether it changes the transition.
- Measure avalanche-size and avalanche-duration distributions using a more rigorous criticality analysis.

# Research summary template

## Title

Write a specific title naming the independent and dependent variables.

## Research question

What relationship did you investigate?

## Hypothesis

State the predicted direction of the effect and explain the mechanism.

## Methods

Include:

- model type;
- parameter changed;
- values tested;
- input protocol: current step or brief network trigger;
- number of neurons and network settings, when relevant;
- measurements calculated.

## Results

Refer to each figure. Describe patterns with numbers when possible.

Avoid writing only “the graph went up.” Instead write something like:

> **Project A example:** Increasing the sodium-conductance scale from 0.75 to 1.25 reduced the smallest tested current that produced a spike from ___ nA to ___ nA.

> **Project B example:** Increasing connection probability from ___ to ___ increased cascade duration from ___ to ___ ms at an E:I ratio of ___.

## Conclusion

State whether the results support the hypothesis and explain the mechanism.

## Limitation

Identify one limitation of the model, parameter sweep, measurements, or simulation design.

## Follow-up experiment

Propose one specific change and predict what it would reveal.

# Troubleshooting and experimental-design checks

### No HH neurons spike in Project A

- Increase the current range gradually.
- Confirm that the current step overlaps the analysis window.
- Check the units: the examples use nA.
- Confirm that gating variables were initialized.

### Project B activity dies immediately in every condition

- Increase connection probability slightly.
- Increase the excitatory fraction or trigger size.
- Increase `excitatory_jump` cautiously.
- Change only one of these settings at a time.

### Project B activity is high in every condition

- Lower connection probability or the excitatory fraction.
- Increase `inhibitory_strength` slightly.
- Reduce the trigger size.
- Confirm that inhibitory neurons have a negative `synaptic_sign`.

### Project B changes when rerun

- Keep `seed_value` fixed for an exact reproduction.
- Treat different seeds as different random networks.
- For a stronger result, repeat each parameter combination across several seeds and report variability.

### Simulations are slow

- Use three rather than five values per independent variable while developing the code.
- Reduce population size temporarily.
- Test the example conditions before launching the full sweep.
- Restore the final settings before making report figures.

### A strong conclusion requires a fair comparison

Change one principal factor at a time. Keep the following fixed unless they are the independent variable:

- simulation duration and numerical time step;
- HH current onset and offset, or the Project B trigger;
- initial conditions and random seed;
- synaptic strengths and delay;
- population size; and
- analysis window and bin size.